In [1]:
import pandas as pd
import numpy as np

# 1. Load the new testing dataset
# Replace the path with your actual filename
df_test_new = pd.read_csv('CSC_CIC_2018Final.csv')

# 2. Basic Inspection
print("--- Dataset Overview ---")
print(f"Total Rows: {df_test_new.shape[0]}")
print(f"Total Columns: {df_test_new.shape[1]}")
print("\nFirst 5 Rows:")
display(df_test_new.head())

# 3. Check for specific problematic values
print("\n--- Value Integrity Check ---")
print(f"Any Null Values? {df_test_new.isnull().values.any()}")
# Checking for infinity which is common in 'Flow Packets/s' or 'Flow Bytes/s'
inf_count = np.isinf(df_test_new.select_dtypes(include=np.number)).values.sum()
print(f"Any Infinity Values? {inf_count > 0} (Count: {inf_count})")

--- Dataset Overview ---
Total Rows: 16232943
Total Columns: 80

First 5 Rows:


,Fwd Pkts/s,Active Min,Protocol,Flow Pkts/s,Bwd PSH Flags,Subflow Fwd Pkts,Idle Min,Bwd Header Len,Active Mean,Bwd URG Flags,...,Pkt Len Max,Tot Bwd Pkts,Flow IAT Std,Fwd Byts/b Avg,RST Flag Cnt,Fwd PSH Flags,Flow IAT Max,Pkt Len Var,Bwd IAT Tot,Bwd Pkts/b Avg
0,0.026633,0.0,0,0.026633,0,3,56320761.0,0,0.0,0,...,0.0,0,139.300036,0,0,0,56320958.0,0.000000,0.0,0
1,0.026633,0.0,0,0.026633,0,3,56320652.0,0,0.0,0,...,0.0,0,114.551299,0,0,0,56320814.0,0.000000,0.0,0
2,0.026634,0.0,0,0.026634,0,3,56319098.0,0,0.0,0,...,0.0,0,301.934596,0,0,0,56319525.0,0.000000,0.0,0
3,2.324152,0.0,6,3.873587,0,15,0.0,328,0.0,0,...,976.0,10,247443.778966,0,0,0,673900.0,77192.153846,5637902.0,0
4,1.590174,0.0,6,2.839597,0,14,0.0,360,0.0,0,...,976.0,11,511356.609733,0,0,0,1928102.0,78267.353846,7715481.0,0



--- Value Integrity Check ---
Any Null Values? True
Any Infinity Values? False (Count: 0)


In [2]:
df_test_new['Label'].value_counts()

Label
Benign                      13484708
DDOS attack-HOIC              686012
DDoS attacks-LOIC-HTTP        576191
DoS attacks-Hulk              461912
Bot                           286191
FTP-BruteForce                193360
SSH-Bruteforce                187589
Infilteration                 161934
DoS attacks-SlowHTTPTest      139890
DoS attacks-GoldenEye          41508
DoS attacks-Slowloris          10990
DDOS attack-LOIC-UDP            1730
Brute Force -Web                 611
Brute Force -XSS                 230
SQL Injection                     87
Name: count, dtype: int64

In [3]:
# Convert Label column to binary
df_test_new['Label'] = df_test_new['Label'].apply(lambda x: 0 if x == 'Benign' else 1)


In [4]:
df_test_new['Label'].value_counts()

Label
0    13484708
1     2748235
Name: count, dtype: int64

In [5]:
import joblib
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model

# 1. Load the Brain
imputer_bundle = joblib.load("regression_imputer.pkl")
scaler = joblib.load("scaler.pkl")


# Load the Random Forest instead of the CNN
rf_model_full = joblib.load("Full_RF_Classifier.pkl") # Or "Full_RF_Classifier.pkl" if you dropped the AE

# 2. Verify the snake_case names
print("--- Features expected by the model ---")
expected_features = list(scaler.feature_names_in_)
print(expected_features)

C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


--- Features expected by the model ---
['dst_port', 'protocol', 'flow_duration', 'tot_fwd_pkts', 'tot_bwd_pkts', 'totlen_fwd_pkts', 'totlen_bwd_pkts', 'fwd_pkt_len_max', 'fwd_pkt_len_min', 'fwd_pkt_len_mean', 'fwd_pkt_len_std', 'bwd_pkt_len_max', 'bwd_pkt_len_min', 'bwd_pkt_len_mean', 'bwd_pkt_len_std', 'flow_byts_s', 'flow_pkts_s', 'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min', 'fwd_iat_tot', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max', 'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean', 'bwd_iat_std', 'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_len', 'bwd_header_len', 'fwd_packets_s', 'bwd_packets_s', 'pkt_len_min', 'pkt_len_max', 'pkt_len_mean', 'pkt_len_std', 'pkt_len_var', 'fin_flag_cnt', 'syn_flag_cnt', 'rst_flag_cnt', 'psh_flag_cnt', 'ack_flag_cnt', 'urg_flag_cnt', 'cwe_flag_count', 'ece_flag_cnt', 'down_up_ratio', 'pkt_size_avg', 'fwd_seg_size_avg', 'bwd_seg_size_avg', 'fwd_byts_b_avg', 'fwd_pkts_b

In [6]:
# Check the raw column names
print("Raw columns in the new dataset:")
print(df_test_new.columns.tolist())

Raw columns in the new dataset:
['Fwd Pkts/s', 'Active Min', 'Protocol', 'Flow Pkts/s', 'Bwd PSH Flags', 'Subflow Fwd Pkts', 'Idle Min', 'Bwd Header Len', 'Active Mean', 'Bwd URG Flags', 'Bwd Pkt Len Min', 'Active Std', 'Bwd IAT Max', 'Fwd Pkts/b Avg', 'CWE Flag Count', 'Fwd Pkt Len Min', 'Init Fwd Win Byts', 'Bwd IAT Mean', 'Pkt Size Avg', 'Idle Std', 'Flow Duration', 'Subflow Bwd Byts', 'Fwd Act Data Pkts', 'Flow IAT Mean', 'PSH Flag Cnt', 'Fwd URG Flags', 'Bwd Blk Rate Avg', 'Pkt Len Mean', 'Idle Mean', 'Bwd Pkts/s', 'Init Bwd Win Byts', 'Flow IAT Min', 'FIN Flag Cnt', 'Fwd Pkt Len Mean', 'Fwd IAT Mean', 'Fwd Seg Size Min', 'Fwd Pkt Len Max', 'Fwd IAT Std', 'Fwd IAT Tot', 'Active Max', 'Bwd Pkt Len Std', 'Subflow Fwd Byts', 'Bwd Byts/b Avg', 'TotLen Bwd Pkts', 'Timestamp', 'Bwd IAT Min', 'Fwd Blk Rate Avg', 'Fwd IAT Min', 'ACK Flag Cnt', 'Pkt Len Min', 'Bwd IAT Std', 'Pkt Len Std', 'Idle Max', 'URG Flag Cnt', 'Subflow Bwd Pkts', 'Fwd Seg Size Avg', 'Fwd IAT Max', 'Fwd Pkt Len Std', 

In [7]:
df_test_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16232943 entries, 0 to 16232942
Data columns (total 80 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Fwd Pkts/s         float64
 1   Active Min         float64
 2   Protocol           int64  
 3   Flow Pkts/s        float64
 4   Bwd PSH Flags      int64  
 5   Subflow Fwd Pkts   int64  
 6   Idle Min           float64
 7   Bwd Header Len     int64  
 8   Active Mean        float64
 9   Bwd URG Flags      int64  
 10  Bwd Pkt Len Min    float64
 11  Active Std         float64
 12  Bwd IAT Max        float64
 13  Fwd Pkts/b Avg     int64  
 14  CWE Flag Count     int64  
 15  Fwd Pkt Len Min    float64
 16  Init Fwd Win Byts  int64  
 17  Bwd IAT Mean       float64
 18  Pkt Size Avg       float64
 19  Idle Std           float64
 20  Flow Duration      int64  
 21  Subflow Bwd Byts   int64  
 22  Fwd Act Data Pkts  int64  
 23  Flow IAT Mean      float64
 24  PSH Flag Cnt       int64  
 25  Fwd URG Flags   

In [8]:
# 1. Strip any leading/trailing whitespace and convert to lowercase
df_test_new.columns = df_test_new.columns.str.strip().str.lower()

# 2. Replace spaces, slashes, and dots with underscores
df_test_new.columns = df_test_new.columns.str.replace(' ', '_', regex=False)
df_test_new.columns = df_test_new.columns.str.replace('/', '_', regex=False)
df_test_new.columns = df_test_new.columns.str.replace('.', '_', regex=False)

# 3. Print the first 10 columns to verify the change
print("First 10 standardized columns:")
print(df_test_new.columns[:10].tolist())

# 4. Find the exact name of your Label column
label_col = [col for col in df_test_new.columns if 'label' in col]
print(f"\nLabel column found: {label_col}")

First 10 standardized columns:
['fwd_pkts_s', 'active_min', 'protocol', 'flow_pkts_s', 'bwd_psh_flags', 'subflow_fwd_pkts', 'idle_min', 'bwd_header_len', 'active_mean', 'bwd_urg_flags']

Label column found: ['label']


In [9]:
# List of columns that are not useful for training features
identifiers = ['timestamp']

# Drop them only if they exist in the current dataframe
df_test_new.drop(columns=[col for col in identifiers if col in df_test_new.columns], inplace=True)

print(f"Identifiers removed.")
print(f"Remaining columns: {df_test_new.shape[1]}")

Identifiers removed.
Remaining columns: 79


In [10]:
# Rename the specific columns to match the training feature set
df_test_new.rename(columns={
    'bwd_iat_tot': 'bwd_iat_total',
    'fwd_pkts_s': 'fwd_packets_s',
    'bwd_pkts_s': 'bwd_packets_s'
}, inplace=True)

# Re-extract X_test_new to ensure it picks up the renamed columns
X_test_new = df_test_new.drop(columns=['label'])

print("Renaming successful.")
print(f"Is 'bwd_iat_tot' now in X? {'bwd_iat_tot' in X_test_new.columns}")
print(f"Is 'fwd_pkts_s' now in X? {'fwd_pkts_s' in X_test_new.columns}")
print(f"Is 'bwd_pkts_s' now in X? {'bwd_pkts_s' in X_test_new.columns}")
print(f"Is 'bwd_iat_total' now in X? {'bwd_iat_total' in X_test_new.columns}")
print(f"Is 'fwd_packets_s' now in X? {'fwd_packets_s' in X_test_new.columns}")
print(f"Is 'bwd_packets_s' now in X? {'bwd_packets_s' in X_test_new.columns}")

Renaming successful.
Is 'bwd_iat_tot' now in X? False
Is 'fwd_pkts_s' now in X? False
Is 'bwd_pkts_s' now in X? False
Is 'bwd_iat_total' now in X? True
Is 'fwd_packets_s' now in X? True
Is 'bwd_packets_s' now in X? True


In [11]:
imputer_models = imputer_bundle['models']
COLS_WONA = imputer_bundle['cols_wona']
COLS_WNA = imputer_bundle['cols_wna']

# Replace Inf with NaN (same as training)
df_test_new.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Remaining NaNs:", df_test_new.isna().sum().sum())

for target_col, model in imputer_models.items():
    if target_col not in df_test_new.columns:
        continue  # safety check

    predict_data = df_test_new[df_test_new[target_col].isna()]

    if not predict_data.empty:
        # Ensure predictor columns exist
        X = predict_data[COLS_WONA]

        predictions = model.predict(X)
        df_test_new.loc[df_test_new[target_col].isna(), target_col] = predictions

print("Remaining NaNs:", df_test_new.isna().sum().sum())

Remaining NaNs: 191520
Remaining NaNs: 0


In [12]:
# 1. Identify rows with Infinity
inf_mask = np.isinf(df_test_new.select_dtypes(include=np.number)).any(axis=1)
inf_count = inf_mask.sum()

# 2. Identify rows with Nulls
null_mask = df_test_new.isnull().any(axis=1)
null_count = null_mask.sum()

print(f"Rows with Infinity: {inf_count}")
print(f"Rows with Nulls: {null_count}")
print(f"Final testing row count: {df_test_new.shape[0]}")

Rows with Infinity: 0
Rows with Nulls: 0
Final testing row count: 16232943


In [13]:
# 1. Extract the labels into y_test_new
y_test_new = df_test_new['label'].copy()

# 2. Extract the features into X_test_new by dropping the label column
X_test_new = df_test_new.drop(columns=['label'])

print("Separation Complete.")
print(f"Features Shape (X): {X_test_new.shape}")
print(f"Labels Shape (y): {y_test_new.shape}")

Separation Complete.
Features Shape (X): (16232943, 78)
Labels Shape (y): (16232943,)


In [14]:
# 1. Get the list of features the scaler expects
expected = list(scaler.feature_names_in_)

# 2. Get the list of features currently in your X_test_new
current = list(X_test_new.columns)

# 3. Find the differences
missing_in_current = [f for f in expected if f not in current]
extra_in_current = [f for f in current if f not in expected]

print("--- Column Name Check ---")
print(f"Features missing from your Data: {missing_in_current}")
print(f"Features in your Data that shouldn't be there: {extra_in_current}")

--- Column Name Check ---
Features missing from your Data: []
Features in your Data that shouldn't be there: []


In [15]:
print("Unique labels found in new data:")
print(y_test_new.unique())

Unique labels found in new data:
[0 1]


In [17]:
print("ALL MATCH:", list(X_test_new.columns) == list(scaler.feature_names_in_))

ALL MATCH: False


In [18]:
# 1. Force exact column alignment (IMPORTANT FIX)
X_test_new = X_test_new.reindex(columns=scaler.feature_names_in_)

# 2. Now transform safely
X_test_scaled = scaler.transform(X_test_new)

# 3. Ensure dtype
X_test_scaled = X_test_scaled.astype('float64')

print("Scaling Complete.")
print(X_test_scaled.shape)

Scaling Complete.
(16232943, 78)


In [19]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Starting Full Random Forest Predictions...")

# 1. Generate direct class predictions from the RF model
# Using the 79 scaled features directly
y_pred = rf_model_full.predict(X_test_scaled) 

# 2. Use the original y_test_new for comparison
print(f"Final Test Accuracy on New Dataset: {accuracy_score(y_test_new, y_pred):.5f}")

print("\nClassification Report:")
# We explicitly set labels=[0, 1] to handle datasets that might only contain attacks
print(classification_report(y_test_new, y_pred, target_names=['BENIGN', 'ATTACK'], labels=[0, 1], zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_new, y_pred, labels=[0, 1]))

Starting Full Random Forest Predictions...


[Parallel(n_jobs=48)]: Using backend ThreadingBackend with 48 concurrent workers.
[Parallel(n_jobs=48)]: Done 100 out of 100 | elapsed:   17.0s finished


Final Test Accuracy on New Dataset: 0.98078

Classification Report:
              precision    recall  f1-score   support

      BENIGN       0.99      0.99      0.99  13484708
      ATTACK       0.94      0.95      0.94   2748235

    accuracy                           0.98  16232943
   macro avg       0.96      0.97      0.97  16232943
weighted avg       0.98      0.98      0.98  16232943


Confusion Matrix:
[[13316379   168329]
 [  143703  2604532]]
